In [ ]:
#@title Cell F4.1 — Overview and fixed Figure 4 design
from IPython.display import display, Markdown

display(Markdown(r"""
# Figure 4 — BMC-N probability calibration (self-contained, v0.5)

**Privacy-minimised public edition.** This notebook embeds only the **60 aggregated reliability-bin rows** calculated from the verified held-out NB55 dataset. It does **not** embed individual BioSample accession numbers, MIC observations, probabilities or any personal file paths.

**Purpose:** reproduce manuscript v0.9 Figure 4 with no external input-file dependency. The underlying NB55 output and full probability calculations are published separately under `03_Results/55_BMC_Probability_Training_Only_Recalibration/`, alongside Notebook 55, for full record-level reproduction.

- **Panel A:** All 18 scenario-specific antibiotic thresholds, 27,315 held-out observations.
- **Panel B:** Cefepime, 1,192 observations.
- **Panel C:** Gentamicin, 5,195 observations.
- Original and recalibrated probabilities use separate ten-bin summaries.
- Error bars show **95% Wilson binomial confidence intervals**.
- Bins with fewer than 10 observations are not plotted.

The embedded aggregated values are checked against the SHA-256 checksum of the previously verified notebook v0.4 summary. This is **plot reproduction from aggregated data**; detailed record-level validation remains in the preceding NB40–55 analysis notebooks. **Do not interpret pooled results as clinically validated susceptibility classifications:** indications are unknown for some scenario-dependent EUCAST thresholds.
"""))


In [ ]:
#@title Cell F4.2 — Load only aggregated plotting data (no paths or patient-level records)
import io
import hashlib
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display

SOURCE_NB55_NAME = '55_BMCN_Probabilities_OuterTest_Original_Vs_Recalibrated.csv.gz'
SOURCE_NB55_SHA256 = 'cee085f109aad69904ebb9a5c003e0441b012c6ac7e43329244d86562b269afb'
SUMMARY_EXPECTED_SHA256 = '0efb6a02b80a3d50f250b18f62e52dbf0ba37f3b51e65e815b9e8c99b48fdfa9'

# Public Figure 4 requires only these 60 aggregated bins, not source-level
# BioSample accessions, measured MICs or held-out prediction records.
EMBEDDED_RELIABILITY_CSV = r"""group,group_label,method,bin_low,bin_high,n,n_exceeds_r,mean_predicted,observed_fraction,wilson_lower95,wilson_upper95,plotted
all_18_descriptive,All 18 antibiotics,Original BMC-N,0.0,0.1,15979,140,0.03087879272649511,0.008761499468051819,0.007429979518196585,0.010329156712492,True
all_18_descriptive,All 18 antibiotics,Original BMC-N,0.1,0.2,2327,112,0.13827540739579264,0.048130640309411256,0.04015408255326536,0.057596649466051,True
all_18_descriptive,All 18 antibiotics,Original BMC-N,0.2,0.3,1207,114,0.24670986820337965,0.09444904722452362,0.07921300109098636,0.11225835722085856,True
all_18_descriptive,All 18 antibiotics,Original BMC-N,0.3,0.4,962,166,0.34986283724589773,0.17255717255717257,0.14999359249802524,0.19772544125598368,True
all_18_descriptive,All 18 antibiotics,Original BMC-N,0.4,0.5,1122,279,0.45007429674392757,0.24866310160427807,0.22425780157534547,0.2747835638130072,True
all_18_descriptive,All 18 antibiotics,Original BMC-N,0.5,0.6,1040,319,0.5509067313950999,0.30673076923076925,0.2794586032149923,0.33542544205547487,True
all_18_descriptive,All 18 antibiotics,Original BMC-N,0.6,0.7,872,404,0.6508903190227693,0.463302752293578,0.4304391027963192,0.4964883115713737,True
all_18_descriptive,All 18 antibiotics,Original BMC-N,0.7,0.8,881,499,0.7497629047395621,0.5664018161180476,0.5334593926920977,0.5987676846375926,True
all_18_descriptive,All 18 antibiotics,Original BMC-N,0.8,0.9,740,554,0.8491088160066956,0.7486486486486487,0.7161644617992446,0.7785646174099687,True
all_18_descriptive,All 18 antibiotics,Original BMC-N,0.9,1.0,2185,2056,0.977925524003228,0.9409610983981693,0.9302828587053809,0.9500915476886895,True
all_18_descriptive,All 18 antibiotics,Recalibrated BMC-N,0.0,0.1,18859,280,0.01638566188205913,0.014847022641709529,0.013217097652715105,0.016674552555532234,True
all_18_descriptive,All 18 antibiotics,Recalibrated BMC-N,0.1,0.2,1723,219,0.14673372948493324,0.12710388856645385,0.11220130202533599,0.1436655330936444,True
all_18_descriptive,All 18 antibiotics,Recalibrated BMC-N,0.2,0.3,1244,240,0.24550073570019315,0.19292604501607716,0.1719572015609105,0.21578553242226822,True
all_18_descriptive,All 18 antibiotics,Recalibrated BMC-N,0.3,0.4,870,289,0.34855049673963984,0.332183908045977,0.30168454136642586,0.36415873368935714,True
all_18_descriptive,All 18 antibiotics,Recalibrated BMC-N,0.4,0.5,650,299,0.44850844813465246,0.46,0.4220321485920805,0.4984378685680577,True
all_18_descriptive,All 18 antibiotics,Recalibrated BMC-N,0.5,0.6,590,355,0.5491797354180691,0.6016949152542372,0.5616576281227128,0.6404165083619766,True
all_18_descriptive,All 18 antibiotics,Recalibrated BMC-N,0.6,0.7,609,434,0.6505641851245035,0.7126436781609196,0.6754581440690085,0.7471633941317525,True
all_18_descriptive,All 18 antibiotics,Recalibrated BMC-N,0.7,0.8,551,425,0.7477840705900743,0.7713248638838476,0.7344503432828886,0.8044423354212575,True
all_18_descriptive,All 18 antibiotics,Recalibrated BMC-N,0.8,0.9,497,432,0.8523442898433764,0.869215291750503,0.8367195097766653,0.8960473040253636,True
all_18_descriptive,All 18 antibiotics,Recalibrated BMC-N,0.9,1.0,1722,1670,0.9723540161613541,0.9698025551684089,0.9606156749119857,0.9768980183247126,True
cefepime,Cefepime,Original BMC-N,0.0,0.1,266,9,0.02669909235547097,0.03383458646616541,0.017900744370341493,0.06304107574932911,True
cefepime,Cefepime,Original BMC-N,0.1,0.2,73,2,0.1501458248718879,0.0273972602739726,0.007545758679607641,0.09450148171690531,True
cefepime,Cefepime,Original BMC-N,0.2,0.3,71,18,0.25408053534392416,0.2535211267605634,0.16680551163187113,0.36553925353561006,True
cefepime,Cefepime,Original BMC-N,0.3,0.4,129,39,0.36258365510694757,0.3023255813953488,0.22973400306410763,0.3863497071514024,True
cefepime,Cefepime,Original BMC-N,0.4,0.5,163,71,0.44933563611401184,0.43558282208588955,0.361814364388415,0.5123176400399994,True
cefepime,Cefepime,Original BMC-N,0.5,0.6,161,90,0.5491294521545754,0.5590062111801242,0.48182385896909463,0.6334384065794478,True
cefepime,Cefepime,Original BMC-N,0.6,0.7,156,95,0.653059724135504,0.6089743589743589,0.5306606636400572,0.6820501077133266,True
cefepime,Cefepime,Original BMC-N,0.7,0.8,118,80,0.7396079770593074,0.6779661016949152,0.5891986209460398,0.7555116312662926,True
cefepime,Cefepime,Original BMC-N,0.8,0.9,43,36,0.839052186807787,0.8372093023255814,0.7002759410625283,0.91883372652938,True
cefepime,Cefepime,Original BMC-N,0.9,1.0,12,9,0.9239418923917914,0.75,0.46769466506643426,0.9110583316059453,True
cefepime,Cefepime,Recalibrated BMC-N,0.0,0.1,277,9,0.026603571387586625,0.032490974729241874,0.017186032838410636,0.06058545756906161,True
cefepime,Cefepime,Recalibrated BMC-N,0.1,0.2,69,3,0.14752886080062452,0.043478260869565216,0.014895903896212308,0.12021202749379145,True
cefepime,Cefepime,Recalibrated BMC-N,0.2,0.3,81,23,0.2482108972391839,0.2839506172839506,0.19728731636228797,0.39017852302208883,True
cefepime,Cefepime,Recalibrated BMC-N,0.3,0.4,163,50,0.35589390076729427,0.3067484662576687,0.24108331947814526,0.38131269380709815,True
cefepime,Cefepime,Recalibrated BMC-N,0.4,0.5,164,85,0.4524979187805751,0.5182926829268293,0.4422805390732324,0.5934674819589837,True
cefepime,Cefepime,Recalibrated BMC-N,0.5,0.6,148,84,0.5498002221390516,0.5675675675675675,0.48704060599477816,0.6446757259960617,True
cefepime,Cefepime,Recalibrated BMC-N,0.6,0.7,156,96,0.6512215078329684,0.6153846153846154,0.5371400173781038,0.6880831522901851,True
cefepime,Cefepime,Recalibrated BMC-N,0.7,0.8,94,67,0.7405312621023795,0.7127659574468085,0.6143682600686844,0.7944563883178923,True
cefepime,Cefepime,Recalibrated BMC-N,0.8,0.9,30,24,0.8420836812718331,0.8,0.6269430358685175,0.9049489282271013,True
cefepime,Cefepime,Recalibrated BMC-N,0.9,1.0,10,8,0.9201951830214543,0.8,0.49016247153664183,0.9433178485456247,True
gentamicin,Gentamicin,Original BMC-N,0.0,0.1,4230,48,0.03641999603398808,0.011347517730496455,0.00856976577051276,0.015012000290350618,True
gentamicin,Gentamicin,Original BMC-N,0.1,0.2,335,12,0.1402882074029189,0.03582089552238806,0.02060774258797044,0.061558881642705794,True
gentamicin,Gentamicin,Original BMC-N,0.2,0.3,151,9,0.24316820000610362,0.059602649006622516,0.03167075659834807,0.10938616050281708,True
gentamicin,Gentamicin,Original BMC-N,0.3,0.4,63,6,0.3422692927452851,0.09523809523809523,0.04438348427689201,0.19261701062836892,True
gentamicin,Gentamicin,Original BMC-N,0.4,0.5,34,8,0.4491707585748489,0.23529411764705882,0.12436985532639575,0.3999613792253638,True
gentamicin,Gentamicin,Original BMC-N,0.5,0.6,11,1,0.5477539700946237,0.09090909090909091,0.016232172885884855,0.37735843645159567,True
gentamicin,Gentamicin,Original BMC-N,0.6,0.7,15,5,0.6519234527143088,0.3333333333333333,0.1517632443971473,0.5828645226148023,True
gentamicin,Gentamicin,Original BMC-N,0.7,0.8,14,9,0.7642575710213263,0.6428571428571429,0.38764423103907397,0.8365526824644587,True
gentamicin,Gentamicin,Original BMC-N,0.8,0.9,29,22,0.8490181754044942,0.7586206896551724,0.5789239838218867,0.8778157755747991,True
gentamicin,Gentamicin,Original BMC-N,0.9,1.0,313,296,0.9890322226635748,0.9456869009584664,0.9147502116084398,0.9658163690355748,True
gentamicin,Gentamicin,Recalibrated BMC-N,0.0,0.1,4704,68,0.01469478293180177,0.01445578231292517,0.011419488574368867,0.018284455376140495,True
gentamicin,Gentamicin,Recalibrated BMC-N,0.1,0.2,101,13,0.1349022805317368,0.12871287128712872,0.07678714443671886,0.20784699892951097,True
gentamicin,Gentamicin,Recalibrated BMC-N,0.2,0.3,20,3,0.23881429337540036,0.15,0.052368745896216595,0.36041886474075696,True
gentamicin,Gentamicin,Recalibrated BMC-N,0.3,0.4,15,5,0.34480247935295905,0.3333333333333333,0.1517632443971473,0.5828645226148023,True
gentamicin,Gentamicin,Recalibrated BMC-N,0.4,0.5,10,7,0.46268166580531433,0.7,0.39677814746114537,0.8922087325936989,True
gentamicin,Gentamicin,Recalibrated BMC-N,0.5,0.6,16,9,0.5423110521991146,0.5625,0.331785563988119,0.7690134759450765,True
gentamicin,Gentamicin,Recalibrated BMC-N,0.6,0.7,16,14,0.6594860581072627,0.875,0.639771727342413,0.9650225122567595,True
gentamicin,Gentamicin,Recalibrated BMC-N,0.7,0.8,11,7,0.7603271563031324,0.6363636363636364,0.35380117450784887,0.8483352890463243,True
gentamicin,Gentamicin,Recalibrated BMC-N,0.8,0.9,28,26,0.8565319582843485,0.9285714285714286,0.7735463743160095,0.9801879358846942,True
gentamicin,Gentamicin,Recalibrated BMC-N,0.9,1.0,274,264,0.9827129651535653,0.9635036496350365,0.9341332084866302,0.9800572117555834,True
"""
SUMMARY_BYTES = EMBEDDED_RELIABILITY_CSV.encode('utf-8')
assert hashlib.sha256(SUMMARY_BYTES).hexdigest() == SUMMARY_EXPECTED_SHA256, (
    'Embedded reliability summary differs from the audited source.')
reliability=pd.read_csv(io.StringIO(EMBEDDED_RELIABILITY_CSV))

OUTPUT_DIR = Path.cwd() / 'Figure04_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
STEM = 'Figure04_BMCN_Probability_Calibration_v0.5'
PNG_FILE = OUTPUT_DIR / (STEM + '.png')
PDF_FILE = OUTPUT_DIR / (STEM + '.pdf')
BINS_FILE = OUTPUT_DIR / (STEM + '_Reliability_Bins.csv')
AUDIT_FILE = OUTPUT_DIR / (STEM + '_Validation.json')
print('Verified 60-row aggregated source; SHA-256:',SUMMARY_EXPECTED_SHA256)
print('Figure outputs will be saved in:',OUTPUT_DIR)


In [ ]:
#@title Cell F4.3 — Audit counts, bin definitions, and absence of individual identifiers
required={'group','group_label','method','bin_low','bin_high','n','n_exceeds_r',
          'mean_predicted','observed_fraction','wilson_lower95','wilson_upper95','plotted'}
assert required.issubset(reliability.columns)
assert len(reliability)==60
assert not reliability.duplicated(['group','method','bin_low']).any()
assert 'biosample' not in reliability.columns
assert not any('patient' in c.lower() or 'interaction_row' in c.lower() for c in reliability.columns)
expected_n={'all_18_descriptive':27315,'cefepime':1192,'gentamicin':5195}
assert set(reliability.group)==set(expected_n)
assert set(reliability.method)=={'Original BMC-N','Recalibrated BMC-N'}
for group,n in expected_n.items():
    for method in ('Original BMC-N','Recalibrated BMC-N'):
        x=reliability[(reliability.group==group)&(reliability.method==method)].sort_values('bin_low')
        assert len(x)==10 and int(x.n.sum())==n
        assert np.allclose(x.bin_low.to_numpy(float), np.arange(10)/10)
        assert np.allclose(x.bin_high.to_numpy(float), np.arange(1,11)/10)
        assert (x.n>=x.n_exceeds_r).all() and (x.n_exceeds_r>=0).all()
        assert (x.loc[x.n.gt(0),'mean_predicted'].between(0,1)).all()
        assert (x.plotted.to_numpy(bool)==x.n.ge(10).to_numpy(bool)).all()
print('Counts and all 60 summary bins verified. No row-level BioSample identifiers are embedded.')


In [ ]:
#@title Cell F4.4 — Independently verify 95% Wilson confidence limits
Z95=1.959963984540054
MIN_N_TO_PLOT=10
METHODS=(('Original BMC-N','prob_original_bmc_n'),
         ('Recalibrated BMC-N','prob_recalibrated'))
GROUPS=(('all_18_descriptive','All 18 antibiotics'),
        ('cefepime','Cefepime'),('gentamicin','Gentamicin'))

def wilson_bounds(n_success,n_total):
    if n_total==0:return np.nan,np.nan
    phat=n_success/n_total
    denominator=1+Z95*Z95/n_total
    midpoint=(phat+Z95*Z95/(2*n_total))/denominator
    half_width=Z95*np.sqrt(phat*(1-phat)/n_total+Z95*Z95/(4*n_total*n_total))/denominator
    return float(max(0,midpoint-half_width)),float(min(1,midpoint+half_width))

for row in reliability.itertuples(index=False):
    if row.n:
        assert abs(row.observed_fraction-row.n_exceeds_r/row.n)<1e-14
        lo,hi=wilson_bounds(row.n_exceeds_r,row.n)
        assert abs(row.wilson_lower95-lo)<1e-14
        assert abs(row.wilson_upper95-hi)<1e-14
    else:
        assert np.isnan(row.observed_fraction) and np.isnan(row.wilson_lower95)
for group,_ in GROUPS:
    n1=int(reliability[(reliability.group==group)&(reliability.method=='Original BMC-N')].n_exceeds_r.sum())
    n2=int(reliability[(reliability.group==group)&(reliability.method=='Recalibrated BMC-N')].n_exceeds_r.sum())
    assert n1==n2
print('Wilson confidence limits and paired group totals independently checked.')
display(reliability[reliability.group.eq('all_18_descriptive')][
    ['method','bin_low','bin_high','n','mean_predicted','observed_fraction']].round(4))


In [ ]:
#@title Cell F4.5 — Draw Figure 4 (three panels, matched manuscript layout)
# Deliberately monochrome: grey dashed open circles for original probabilities,
# solid black squares for recalibrated probabilities, grey dotted ideal line.
# Wilson intervals are displayed for both methods.

plt.rcParams.update({
    'font.family':'DejaVu Sans',
    'axes.linewidth':1.2,
    'pdf.fonttype':42,
    'ps.fonttype':42,
    'savefig.facecolor':'white'
})
fig,axes=plt.subplots(1,3,figsize=(13.07,5.11),sharex=True,sharey=True)
fig.subplots_adjust(left=.110,right=.972,bottom=.270,top=.862,wspace=.37)

for ax,(panel,(group,title)) in zip(axes,zip('ABC',GROUPS)):
    ax.plot([0,1],[0,1],color='0.63',linestyle=':',linewidth=1.65,zorder=1)
    for method,color,style,marker,filled,zorder in [
        ('Original BMC-N','0.46','--','o',False,3),
        ('Recalibrated BMC-N','black','-','s',True,4),
    ]:
        t=reliability.loc[(reliability.group==group)&
                          (reliability.method==method)&
                          reliability.plotted].sort_values('bin_low')
        x=t.mean_predicted.to_numpy(float)
        y=t.observed_fraction.to_numpy(float)
        low=t.wilson_lower95.to_numpy(float)
        high=t.wilson_upper95.to_numpy(float)
        ax.errorbar(x,y,yerr=np.vstack([np.maximum(0,y-low),np.maximum(0,high-y)]),
                    color=color, ecolor=color, linestyle=style, marker=marker,
                    markersize=5.4, markerfacecolor=(color if filled else 'white'),
                    markeredgecolor=color, markeredgewidth=1.2,
                    linewidth=1.5, elinewidth=1.1, capsize=2.1, capthick=1.1,
                    zorder=zorder)
    ax.set_xlim(0,1);ax.set_ylim(0,1)
    ax.set_xticks(np.linspace(0,1,6));ax.set_yticks(np.linspace(0,1,6))
    ax.set_xticklabels([f'{x:.1f}' for x in np.linspace(0,1,6)])
    ax.set_yticklabels([f'{y:.1f}' for y in np.linspace(0,1,6)])
    ax.tick_params(axis='both',labelsize=12.8,length=5.2,width=1.15,pad=6)
    ax.set_title(f'{panel}  {title}',loc='left',fontsize=14.7,pad=11)
    ax.text(.965,.035,f'n = {expected_n[group]:,}',
            ha='right',va='bottom',color='0.36',fontsize=12.8,transform=ax.transAxes)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.supylabel('Observed threshold-exceedance frequency',x=.047,fontsize=14.5)
fig.supxlabel('Mean predicted threshold-exceedance probability',y=.137,fontsize=14.5)

legend_handles=[
    Line2D([0],[0],linestyle='--',color='0.46',marker='o',markersize=5.4,
           markerfacecolor='white',markeredgewidth=1.2,linewidth=1.5,label='Original BMC-N'),
    Line2D([0],[0],linestyle='-',color='black',marker='s',markersize=5.4,
           markerfacecolor='black',linewidth=1.5,label='Recalibrated BMC-N'),
    Line2D([0],[0],linestyle=':',color='0.63',linewidth=1.65,label='Ideal calibration')
]
fig.legend(handles=legend_handles,loc='lower center',bbox_to_anchor=(.51,.041),
           ncol=3,frameon=False,fontsize=13.3,handlelength=2.8,columnspacing=1.5)

fig.savefig(PNG_FILE,dpi=150,facecolor='white')
fig.savefig(PDF_FILE,facecolor='white')
plt.show()
print('Saved PNG:',PNG_FILE)
print('Saved PDF:',PDF_FILE)


In [ ]:
#@title Cell F4.6 — Export audited aggregate statistics and provenance
reliability.to_csv(BINS_FILE,index=False)
audit={
    'source_notebook':'55_BMC_Probability_Training_Only_Recalibration_v0.1',
    'source_data_filename':SOURCE_NB55_NAME,
    'source_data_sha256':SOURCE_NB55_SHA256,
    'input_mode':'60 verified aggregated bins embedded, no individual records or paths',
    'summary_source_sha256':SUMMARY_EXPECTED_SHA256,
    'manuscript_figure':'Figure 4, BMC manuscript v0.9',
    'n_heldout_total':expected_n['all_18_descriptive'],
    'panel_counts':expected_n,
    'n_bins_per_group_and_method':10,
    'min_bin_n_for_plot':MIN_N_TO_PLOT,
    'wilson_z':Z95,
    'pooled_observed_exceedances':int(reliability[(reliability.group=='all_18_descriptive')&
                              (reliability.method=='Original BMC-N')].n_exceeds_r.sum()),
    'raw_probabilities':'Not embedded in the public figure notebook. See the original NB55 dataset and Notebook 55.',
    'outcome':'observed MIC > scenario-specific EUCAST resistance cutoff',
    'figure_outputs':[PNG_FILE.name,PDF_FILE.name],
    'reliability_bins_output':BINS_FILE.name,
}
AUDIT_FILE.write_text(json.dumps(audit,indent=2)+'\n',encoding='utf-8')
assert all(p.is_file() and p.stat().st_size>0 for p in [PNG_FILE,PDF_FILE,BINS_FILE,AUDIT_FILE])
print('Saved bin audit:',BINS_FILE)
print('Saved validation metadata:',AUDIT_FILE)
print('Figure 4 reconstruction complete; no patient-level records were embedded.')
